# Apollo 11 source-to-answer walkthrough

Use the external inputs listed in the [walkthrough guide](README.md). The local Ollama step is optional.

In [ ]:
from pathlib import Path
import sys

sys.dont_write_bytecode = True
here = Path.cwd().resolve()
if here.name == 'apollo' and here.parent.name == 'examples':
    sys.path.insert(0, str(here.parents[1]))

from examples.apollo.demo import QUESTION, compile_model, project_model, prepare_prompt, generate_local
from sysml_graphrag.io import write

source = Path('<Apollo source checkout>')
pilot = Path('<extracted SysML Pilot sysml directory>')
encoder = Path('<local MiniLM snapshot>')
tokenizer = Path('<local Llama 3.1 chat tokenizer>')
out = Path('<new external output directory>')

In [ ]:
export = compile_model(source, pilot, out)
graph = project_model(export)
print(f'{len(graph):,} projected nodes, {graph.number_of_edges():,} edges')
print(out / 'graph.json')

In [ ]:
row = prepare_prompt(graph, encoder, tokenizer, QUESTION)
write(out / 'prompt.json', row)
print(row['query'])
print(row['router_result']['route'], row['renderer'])
print(row['prompt_fit'])
print(row['packed']['evidence'] if row['generation_planned'] else row['prompt_fit']['fixed_response'])

In [ ]:
row['packed']['messages'] if row['generation_planned'] else []

In [ ]:
GENERATE = False
if GENERATE:
    response = generate_local(row)
    write(out / 'answer.json', response)
    print(response['answer'])
    print({key: value for key, value in response.items() if key != 'answer'})